In [ ]:
import os
from dotenv import load_dotenv
from sqlalchemy import create_engine
import matplotlib as mpl

load_dotenv("../.env", override=True)

db_host = os.getenv("DB_HOST", "localhost")
db_port = os.getenv("DB_PORT", "5432")
db_name = os.getenv("DB_NAME", "postgres")
db_user = os.getenv("DB_USER", "postgres")
db_password = os.getenv("DB_PASSWORD", "postgres")

connection_string = f"postgresql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
conn = create_engine(connection_string)

mpl.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Linux Libertine', 'Libertine', 'Linux Libertine O', 'Times New Roman', 'Times'],
    'axes.facecolor': 'white',
    'figure.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.linewidth': 1.2,
    'grid.color': '#cccccc',
    'grid.linestyle': '--',
    'grid.linewidth': 0.7,
    #'legend.frameon': False,
    'axes.grid': True,
    'axes.axisbelow': True,
    'savefig.dpi': 300,
    'savefig.format': 'pdf',
    'pdf.fonttype': 42,
    'ps.fonttype': 42
})

## Number of Mutants per Project

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT mr.project_id, mr.project_name, mr.variant, sum(mr.total) AS total
FROM mv_mutation_results_by_project_variant_mutator mr
JOIN v_projects_successes ps ON ps.project_id = mr.project_id
WHERE mr.variant IN ('ORIGINAL', 'INITIAL')
GROUP BY mr.project_id, mr.project_name, mr.variant
ORDER BY mr.project_id, variant_order(mr.variant)
""", conn)

df

## Number of Mutants per Project + Mutator

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT mr.project_id, mr.project_name, mr.variant, mr.mutator, mr.total
FROM mv_mutation_results_by_project_variant_mutator mr
JOIN v_projects_successes ps ON ps.project_id = mr.project_id
WHERE mr.variant = 'INITIAL'
""", conn)

df

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def create_mutator_bar_chart(data):
    # Sort by project_id
    data = data.sort_values(['project_id'])

    # Create figure with adjusted size to accommodate the legend
    plt.figure(figsize=(18, 6))

    # Create combined project identifier (id + name)
    data['project_label'] = data['project_id'].astype(str) + ': ' + data['project_name']

    # Get unique project labels and mutators
    project_labels = data['project_label'].unique()
    mutators = sorted(data['mutator'].unique())  # Sort mutators for consistent indexing

    # Create mutator indices and labels for legend
    mutator_indices = {mutator: f"[{i+1}]" for i, mutator in enumerate(mutators)}
    mutator_legend_labels = [f"{idx} {mutator}" for mutator, idx in mutator_indices.items()]

    # Create a color map for mutators
    color_map = plt.colormaps['tab10']
    mutator_colors = {mutator: color_map(i % 10) for i, mutator in enumerate(mutators)}

    # Set up the plot
    ax = plt.subplot(111)
    bar_width = 0.8 / len(mutators)

    # For each project, plot bars for each mutator
    for i, project_label in enumerate(project_labels):
        project_data = data[data['project_label'] == project_label]

        # For each mutator, find its data for this project
        for j, mutator in enumerate(mutators):
            mutator_data = project_data[project_data['mutator'] == mutator]

            # Calculate bar position
            x_pos = i + (j * bar_width) - (len(mutators) * bar_width / 2) + (bar_width / 2)

            # If we have data for this mutator in this project
            if not mutator_data.empty:
                value = mutator_data['total'].values[0]

                # Plot the bar with consistent color
                bar = ax.bar(x_pos, value, width=bar_width, 
                       color=mutator_colors[mutator],
                       label=mutator_legend_labels[j] if i == 0 else "")

                # Add value on top of the bar
                ax.text(x_pos, value + 0.1, str(int(value)), 
                        ha='center', va='bottom', fontsize=9)
            else:
                # Plot an empty/zero bar to maintain spacing
                ax.bar(x_pos, 0, width=bar_width, color=mutator_colors[mutator],
                      label=mutator_legend_labels[j] if i == 0 else "")

    # Set x-axis labels and ticks
    ax.set_xlabel('Project')
    ax.set_ylabel('Number of Mutations')
    ax.set_title('Mutations by Project and Mutator')
    ax.set_xticks(range(len(project_labels)))
    ax.set_xticklabels(project_labels)
    ax.tick_params(axis='x', which='major', pad=15)

    # Add mutator indices below the x-axis
    for i, project_label in enumerate(project_labels):
        for j, mutator in enumerate(mutators):
            x_pos = i + (j * bar_width) - (len(mutators) * bar_width / 2) + (bar_width / 2)
            # Add index at a fixed position below the x-axis
            ax.text(x_pos, -50, mutator_indices[mutator], 
                    ha='center', va='top', fontsize=9, fontweight='bold')

    # Create legend with unique entries and position it outside the plot
    handles, labels = plt.gca().get_legend_handles_labels()
    plt.legend(handles, labels, 
               title='Mutator',
               loc='center left', 
               bbox_to_anchor=(1.0, 0.5))

    # Add a bit of padding to the top and bottom to accommodate the numbers and indices
    y_max = data['total'].max() if not data.empty else 10
    plt.ylim(-10, y_max * 1.1)  # Fixed bottom margin for indices

    # Adjust layout to make room for the legend
    plt.tight_layout()
    plt.subplots_adjust(right=0.85, bottom=0.2)  # Adjust margins for legend and indices

    return plt

# Create the plot
plot = create_mutator_bar_chart(df)
plot.show()

In [ ]:
# Group by project_id and calculate the total mutants per project
project_totals = df.groupby('project_id')['total'].sum().reset_index()
project_totals.rename(columns={'total': 'project_total'}, inplace=True)

# Merge the project totals back to the original dataframe
df = pd.merge(df, project_totals, on='project_id')

# Calculate the percentage
df['mutator_percentage'] = (df['total'] / df['project_total'] * 100).round(2)

# Display the result
df

## Number of (Detected) Mutants per Mutator

In [ ]:
import pandas as pd
import numpy as np

# Query to get mutator data for all variants
query = """
SELECT
    mr.mutator,
    mr.variant,
    SUM(mr.total) as total_mutants,
    SUM(mr.covered) as covered_mutants,
    SUM(mr.detected) as detected_mutants
FROM mv_mutation_results_by_project_variant_mutator mr
JOIN v_projects_successes ps ON ps.project_id = mr.project_id
WHERE mr.variant IN ('INITIAL', 'NAIVE_200_TRIES', 'IMPROVED_200_TRIES')
GROUP BY mr.mutator, mr.variant
"""
variant_data = pd.read_sql_query(query, conn)

# Query to get project-level data for calculating min/max percentages
project_query = """
SELECT
    mr.project_id,
    mr.mutator,
    mr.total
FROM mv_mutation_results_by_project_variant_mutator mr
JOIN v_projects_successes ps ON ps.project_id = mr.project_id
WHERE mr.variant = 'INITIAL'
"""
project_data = pd.read_sql_query(project_query, conn)

# Calculate total mutants per project
project_totals = project_data.groupby('project_id')['total'].sum().reset_index()
project_totals.rename(columns={'total': 'project_total'}, inplace=True)

# Merge to get project totals
project_data = pd.merge(project_data, project_totals, on='project_id')

# Calculate percentage of each mutator within each project
project_data['project_percent'] = (project_data['total'] / project_data['project_total'] * 100)

# Calculate min and max percentages for each mutator
min_max_percent = project_data.groupby('mutator')['project_percent'].agg(['min', 'max']).reset_index()
min_max_percent.columns = ['mutator', 'min_percent', 'max_percent']

# Filter for INITIAL variant to calculate overall percentages
initial_data = variant_data[variant_data['variant'] == 'INITIAL']
total_initial_mutants = initial_data['total_mutants'].sum()

# Calculate overall percentage for each mutator
initial_summary = initial_data.copy()
initial_summary['percent'] = (initial_summary['total_mutants'] / total_initial_mutants * 100)

# Calculate detected percentage for each variant
variant_data['detected_of_covered_pct'] = (
    variant_data['detected_mutants'] / variant_data['covered_mutants'] * 100
).fillna(0)  # Handle division by zero

# Pivot the variant data to get columns for each variant
variant_pivot = variant_data.pivot(
    index='mutator',
    columns='variant',
    values='detected_of_covered_pct'
).reset_index()

# Rename the columns
variant_pivot.columns.name = None
variant_pivot = variant_pivot.rename(columns={
    'INITIAL': 'detected_pct_initial',
    'NAIVE_200_TRIES': 'detected_pct_naive',
    'IMPROVED_200_TRIES': 'detected_pct_improved'
})

# Calculate the percentage point differences
variant_pivot['detected_diff_naive'] = (
    variant_pivot['detected_pct_naive'] - variant_pivot['detected_pct_initial']
)
variant_pivot['detected_diff_improved'] = (
    variant_pivot['detected_pct_improved'] - variant_pivot['detected_pct_initial']
)

# Create the final result by merging all the data
result = pd.merge(
    initial_summary[['mutator', 'total_mutants', 'percent']],
    min_max_percent,
    on='mutator'
)

result = pd.merge(
    result,
    variant_pivot,
    on='mutator'
)

# Rename columns for clarity
result = result.rename(columns={'total_mutants': 'total'})

# Reorder the columns as requested
result = result[['mutator', 'total', 'percent', 'min_percent', 'max_percent',
                'detected_pct_initial', 'detected_pct_naive', 'detected_diff_naive',
                'detected_pct_improved', 'detected_diff_improved']]

# Sort by total mutants in descending order
result = result.sort_values('total', ascending=False)

# Display the result
display(result)


In [ ]:
# Copy and normalize the data
df_filtered = result.copy()

# Clean up mutator names
df_filtered['mutator'] = df_filtered['mutator'].str.replace('Mutator', '').str.strip()
df_filtered['mutator'] = df_filtered['mutator'].str.replace('RemoveConditional_ORDER_ELSE', 'RemoveConditionalOrderElse').str.strip()
df_filtered['mutator'] = df_filtered['mutator'].str.replace('RemoveConditional_EQUAL_ELSE', 'RemoveConditionalEqualElse').str.strip()

# Select the columns
columns_to_keep = [
    'mutator',
    'total',
    'percent',
    'min_percent',
    'max_percent',
    'detected_pct_initial',
    'detected_pct_naive',
    'detected_diff_naive',
    'detected_pct_improved',
    'detected_diff_improved'
]
df_filtered = df_filtered[columns_to_keep]

# Ensure total is integer
df_filtered['total'] = df_filtered['total'].astype(int)

# Format all percentage columns to 2 decimals
for col in ['detected_pct_initial', 'detected_pct_naive', 'detected_pct_improved', 'percent', 'min_percent', 'max_percent']:
    df_filtered[col] = df_filtered[col].astype(float).round(2)

# Begin LaTeX table
latex_table = """\\begin{table}[H]
  \\caption{Number of mutants and percentage of detections per mutator.}
  \\label{tab:detections-per-mutator}
  \\begin{tabular}{lrrrrcrrrr}
    \\toprule
    & & & & & \\multicolumn{5}{c}{Detected \\%} \\\\
    \\cmidrule{6-10}
    Mutator & Total & Total \\% & Min \\% & Max \\% & INITIAL & \\multicolumn{2}{c}{NAIVE$_{200}$} & \\multicolumn{2}{c}{IMPROVED$_{200}$} \\\\
    \\midrule"""

# Generate table rows
for _, row in df_filtered.iterrows():
    naive_val = row['detected_pct_naive']
    improved_val = row['detected_pct_improved']
    naive_diff = row['detected_diff_naive']
    improved_diff = row['detected_diff_improved']

    naive_diff_str = (
        f"(+{naive_diff:.2f})" if naive_diff > 0
        else f"({naive_diff:.2f})" if naive_diff < 0
        else "--"
    )
    improved_diff_str = (
        f"(+{improved_diff:.2f})" if improved_diff > 0
        else f"({improved_diff:.2f})" if improved_diff < 0
        else "--"
    )

    latex_table += (
        f"\n    {row['mutator']} & {row['total']} & {row['percent']:.2f} & "
        f"{row['min_percent']:.2f} & {row['max_percent']:.2f} & {row['detected_pct_initial']:.2f} & "
        f"{naive_val:.2f} & {naive_diff_str} & {improved_val:.2f} & {improved_diff_str} \\\\"
    )

# Close the table
latex_table += """
    \\bottomrule
  \\end{tabular}
\\end{table}"""

# Display the LaTeX code
print(latex_table)


## Un-/Covered Mutants per Project + Variant

In [ ]:
from natsort import natsorted
import pandas as pd
import re

df = pd.read_sql_query("""
SELECT mr.project_id, mr.project_name AS project, mr.variant, mr.total, mr.covered, mr.uncovered, mr.covered_pct, mr.uncovered_pct
FROM mv_mutation_results_by_project_variant mr
JOIN v_projects_successes ps ON ps.project_id = mr.project_id
WHERE mr.variant = 'INITIAL'
""", conn)

df = df.drop(columns=['variant'])
df = df.reindex(index=natsorted(df.index, key=lambda x: df.loc[x, 'project']))

# total tests
df_total_tests = pd.read_sql_query("""
SELECT t.project_id, project_name(t.project_id) AS project, count(*) AS total_tests
FROM test t
JOIN project p ON t.project_id = p.id
JOIN v_projects_successes ps ON ps.project_id = t.project_id
WHERE p.use_test_generalization
GROUP BY t.project_id;
""", conn)

# included tests
df_included_tests = pd.read_sql_query("""
SELECT r.project_id, project_name(r.project_id) AS project, count(*) AS included_tests
FROM junit_test_report r
JOIN v_projects_successes ps ON ps.project_id = r.project_id
WHERE r.stage = 'COLLECT_JUNIT_REPORTS_INITIAL'
GROUP BY r.project_id;
""", conn)

# total classes
df_projects = pd.read_sql_query("""
SELECT p.id AS project_id, project_name(p.id), '../' || p.main_source_path AS test_source_directory
FROM project p
JOIN v_projects_successes ps ON ps.project_id = p.id
WHERE p.use_test_generalization
""", conn)

def collect_stats(directory):
    num_files = 0
    num_classes = 0

    if not os.path.exists(directory):
        print(f"Directory does not exist: {directory}")
        return {"total_files": 0, "total_classes": 0}

    for root, _, files in os.walk(directory):
        for file in files:
            if file.endswith(".java"):
                num_files += 1
                file_path = os.path.join(root, file)
                try:
                    with open(file_path, encoding="utf-8", errors="ignore") as f:
                        content = f.read()
                        num_classes += len(re.findall(r'\b(class|enum)\s+\w+', content))
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")

    if num_files == 0:
        print(f"No .java files found in {directory}")

    return {
        "total_files": num_files,
        "total_classes": num_classes,
    }

stats_df = df_projects['test_source_directory'].apply(collect_stats).apply(pd.Series)
df_total_classes = pd.concat([df_projects, stats_df[['total_files', 'total_classes']]], axis=1)

# included classes
df_included_classes = pd.read_sql_query("""
SELECT r.project_id, project_name(r.project_id) AS project, count(*) AS included_classes
FROM jacoco_coverage_report r
JOIN v_projects_successes ps ON ps.project_id = r.project_id
WHERE r.stage = 'COLLECT_JACOCO_DATA_INITIAL' AND instruction_covered > 0 AND covered_class NOT LIKE '%%...%%'
GROUP BY r.project_id;
""", conn)

# Now, merge all DataFrames on 'project_id'
df = df.merge(df_total_tests[['project_id', 'total_tests']], on='project_id', how='left')
df = df.merge(df_included_tests[['project_id', 'included_tests']], on='project_id', how='left')
df = df.merge(df_total_classes[['project_id', 'total_classes']], on='project_id', how='left')
df = df.merge(df_included_classes[['project_id', 'included_classes']], on='project_id', how='left')

# Display the merged DataFrame
display(df)

# Generate and display the LaTeX table
def get_base_project(name):
    return name.split('-es-')[0]

# Build LaTeX table
latex_table = r"""\begin{table}[H]
  \caption{Number of total, covered, and uncovered mutants in included classes per project.}
  \label{tab:mutants-per-project}
  \begin{tabular}{lrrrrr}
    \toprule
            & Included     & Included      & \multicolumn{3}{r}{Mutants} \\
                                             \cmidrule(lr){4-6}
    Project & Test Methods & Impl. Classes & Total & Covered & Uncovered \\
    \midrule
"""

prev_base = None
for _, row in df.iterrows():
    base = get_base_project(row["project"])
    if prev_base is not None and base != prev_base:
        latex_table += "    \\midrule\n"
    prev_base = base
    latex_table += (
        f"    {row['project']} & "
        f"{row['included_tests']}\\; ({row['included_tests'] * 100.0 / row['total_tests']:.1f} \\%) & "
        f"{row['included_classes']}\\; ({row['included_classes'] * 100.0 / row['total_classes']:.1f} \\%) & "
        f"{row['total']} & "
        f"{row['covered']}\\; ({'\\phantom{0}' if row['covered_pct'] < 10 else ''}{row['covered_pct']:.1f} \\%) & "
        f"{row['uncovered']}\\; ({'\\phantom{0}' if row['uncovered_pct'] < 10 else ''}{row['uncovered_pct']:.1f} \\%) \\\\\n"
    )

latex_table += r"""    \bottomrule
  \end{tabular}
\end{table}
"""

print(latex_table)

## Percentage of Survived / Detected / ... Mutants per Project + Variant

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT
    mr.project_id, mr.project_name, mr.variant,
    mr.survived_of_covered_pct, mr.detected_of_covered_pct, mr.killed_of_covered_pct, mr.timed_out_of_covered_pct, mr.memory_error_of_covered_pct, mr.run_error_of_covered_pct,
    mr.survived_of_covered_pct_diff, mr.detected_of_covered_pct_diff, mr.killed_of_covered_pct_diff, mr.timed_out_of_covered_pct_diff, mr.memory_error_of_covered_pct_diff, mr.run_error_of_covered_pct_diff
FROM mv_mutation_results_by_project_variant mr
JOIN v_projects_successes ps ON ps.project_id = mr.project_id
WHERE mr.variant NOT IN ('ORIGINAL', 'BASELINE')
""", conn)

df = df.rename(columns={
    'survived_of_covered_pct': 'Survived (%)',
    'detected_of_covered_pct': 'Detected (%)',
    'killed_of_covered_pct': 'Killed (%)',
    'timed_out_of_covered_pct': 'Timed-Out (%)',
    'memory_error_of_covered_pct': 'Memory Error (%)',
    'run_error_of_covered_pct': 'Run Error (%)',
    'survived_of_covered_pct_diff': 'Survived (%) Diff.',
    'detected_of_covered_pct_diff': 'Detected (%) Diff.',
    'killed_of_covered_pct_diff': 'Killed (%) Diff.',
    'timed_out_of_covered_pct_diff': 'Timed-Out (%) Diff.',
    'memory_error_of_covered_pct_diff': 'Memory Error (%) Diff.',
    'run_error_of_covered_pct_diff': 'Run Error (%) Diff.',
})

df

In [ ]:
import matplotlib.pyplot as plt
import re
from matplotlib.ticker import PercentFormatter
from matplotlib.gridspec import GridSpec
import numpy as np

# Natural sort function
def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

# Get unique projects and sort them naturally
projects = sorted(df['project_name'].unique(), key=natural_sort_key)

# Increase font sizes globally
plt.rcParams.update({
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 14,
    'xtick.labelsize': 12,
    'ytick.labelsize': 12,
    'legend.fontsize': 12
})

# Create figure with custom grid layout
fig = plt.figure(figsize=(15, 12))
gs = GridSpec(3, 3, figure=fig)

# Create the axes with the desired layout
axes = []
# First row with 1 centered subplot
axes.append(fig.add_subplot(gs[0, 1]))  # Center position in first row
# Second row with 3 subplots
axes.append(fig.add_subplot(gs[1, 0]))
axes.append(fig.add_subplot(gs[1, 1]))
axes.append(fig.add_subplot(gs[1, 2]))
# Third row with 3 subplots
axes.append(fig.add_subplot(gs[2, 0]))
axes.append(fig.add_subplot(gs[2, 1]))
axes.append(fig.add_subplot(gs[2, 2]))

# Create an axis for the legend in the top-right empty cell
legend_ax = fig.add_subplot(gs[0, 2])
legend_ax.axis('off')  # Hide the axis

# Use the first three colors from the tab10 palette
tab10_colors = plt.cm.tab10.colors
palette = {
    'INITIAL': tab10_colors[0],  # Blue
    'NAIVE': tab10_colors[1],    # Orange
    'IMPROVED': tab10_colors[2]  # Green
}

# Loop through projects
for i, project in enumerate(projects):
    if i < len(axes):  # Ensure we don't exceed the number of subplots
        # Filter data for this project
        project_data = df[df['project_name'] == project]

        # Sort variants
        variants = project_data['variant'].tolist()

        # Create x positions for the bars
        x_positions = np.arange(len(variants))

        # Determine color for each variant
        bar_colors = []
        for variant in variants:
            if variant == 'INITIAL':
                bar_colors.append(palette['INITIAL'])
            elif variant.startswith('NAIVE'):
                bar_colors.append(palette['NAIVE'])
            elif variant.startswith('IMPROVED'):
                bar_colors.append(palette['IMPROVED'])
            else:
                bar_colors.append('gray')  # Default color

        # Plot bars with tab10 colors
        bars = axes[i].bar(x_positions, project_data['Detected (%)'],
                          color=bar_colors, width=0.6)

        # Add value labels on top of bars
        for bar in bars:
            height = bar.get_height()
            axes[i].text(bar.get_x() + bar.get_width()/2., height + 1,
                        f'{height:.1f}%', ha='center', va='bottom', fontsize=11)

        # Set title and labels
        axes[i].set_title(project, fontweight='bold')
        axes[i].set_ylim(0, 100)  # Percentage scale
        axes[i].yaxis.set_major_formatter(PercentFormatter())

        # Set the tick positions and labels properly
        axes[i].set_xticks(x_positions)

        # Function to transform variant names
        def transform_variant_name(variant):
            # Regular expression to match _X_TRIES pattern
            pattern = r'(.+)_(\d+)_TRIES$'
            match = re.match(pattern, variant)

            if match:
                base_name = match.group(1)
                number = match.group(2)
                # Create the name with subscript
                return f"{base_name}$_{{{number}}}$"
            else:
                return variant

        # Modified code for setting xticklabels
        transformed_variants = [transform_variant_name(variant) for variant in variants]
        axes[i].set_xticklabels(transformed_variants, rotation=45, ha='right')

        # Add grid lines for readability
        axes[i].grid(axis='y', linestyle='--', alpha=0.7)

# Add a common y-label
fig.text(0.04, 0.5, 'Detected Mutants (%)', va='center', rotation='vertical', fontsize=16, fontweight='bold')

# Add a legend in the top-right empty cell
handles = [plt.Rectangle((0,0),1,1, color=color) for color in palette.values()]
legend_ax.legend(handles, ['INITIAL', 'NAIVE Variants', 'IMPROVED Variants'],
                loc='center', fontsize=14)

plt.tight_layout(rect=[0.05, 0.03, 1, 0.95])

# Save figure as PDF for scientific paper
plt.savefig('fig_mutation_detection_results.pdf', format='pdf', dpi=300, bbox_inches='tight')

plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import re
from matplotlib.patches import Patch

tab10 = plt.get_cmap("tab10")
colors = [tab10(i) for i in range(3)]  # blue, orange, green

def get_base_project(name):
    return re.sub(r'-\d+s$', '', name)

def natural_sort_key(s):
    return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

def format_variant_label(label):
    match = re.match(r'([A-Z]+)_(\d+)_TRIES', label)
    if match:
        name, number = match.groups()
        return f"{name}$_{{{number}}}$"
    else:
        return label

def get_variant_color(variant):
    if variant == "INITIAL":
        return colors[0]  # blue
    elif variant.startswith("NAIVE"):
        return colors[1]  # orange
    elif variant.startswith("IMPROVED"):
        return colors[2]  # green
    else:
        return "#bbbbbb"  # gray for unknown

df['base_project'] = df['project_name'].apply(get_base_project)
projects = sorted(df['project_name'].unique(), key=natural_sort_key)
base_projects = sorted(df['base_project'].unique(), key=natural_sort_key)

improvement_range = {}
for base in base_projects:
    max_improvement = 0
    for project in df[df['base_project'] == base]['project_name'].unique():
        project_data = df[df['project_name'] == project]
        initial_row = project_data[project_data['variant'] == 'INITIAL']
        improvement_data = project_data[project_data['variant'] != 'INITIAL']
        if not initial_row.empty and not improvement_data.empty:
            initial_detected = initial_row['Detected (%)'].iloc[0]
            improvement = improvement_data['Detected (%)'] - initial_detected
            if not improvement.empty:
                max_improvement = max(max_improvement, improvement.max())
    margin = 0.45 * abs(max_improvement)
    improvement_range[base] = (0, max_improvement + margin)

n_projects = len(projects)
fig, axes = plt.subplots(n_projects, 2, figsize=(12, 0.2 + 2 * n_projects), squeeze=False)

for i, project in enumerate(projects):
    project_data = df[df['project_name'] == project].copy()
    variants = project_data['variant'].tolist()
    x_positions = np.arange(len(variants))

    bar_colors_left = [get_variant_color(v) for v in variants]

    # --- Left plot: Detection rates (all variants) ---
    bars_left = axes[i, 0].bar(x_positions, project_data['Detected (%)'], color=bar_colors_left)
    axes[i, 0].set_title(f"{project}")
    axes[i, 0].set_ylabel('Detected (%)')
    axes[i, 0].set_ylim(0, 100)
    axes[i, 0].grid(False)

    for bar in bars_left:
        height = bar.get_height()
        axes[i, 0].text(bar.get_x() + bar.get_width()/2, height + 1, f'{height:.2f}', ha='center', va='bottom', fontsize=12)

    # --- Right plot: Detection rate improvement (exclude INITIAL) ---
    initial_row = project_data[project_data['variant'] == 'INITIAL']
    improvement_data = project_data[project_data['variant'] != 'INITIAL'].copy()
    improvement_variants = improvement_data['variant'].tolist()
    x_positions_imp = np.arange(len(improvement_variants))
    if not initial_row.empty and not improvement_data.empty:
        initial_detected = initial_row['Detected (%)'].iloc[0]
        improvement = improvement_data['Detected (%)'] - initial_detected
    else:
        improvement = [0] * len(improvement_data)

    bar_colors_right = [get_variant_color(v) for v in improvement_variants]

    bars_right = axes[i, 1].bar(x_positions_imp, improvement, color=bar_colors_right)
    axes[i, 1].axhline(0, color='gray', linewidth=0.8)
    axes[i, 1].set_title(f"{project}")
    axes[i, 1].set_ylabel('Improvement (%)')
    y_min, y_max = improvement_range[get_base_project(project)]
    axes[i, 1].set_ylim(y_min, y_max)
    axes[i, 1].grid(False)

    for j, bar in enumerate(bars_right):
        height = bar.get_height()
        if not initial_row.empty and not improvement_data.empty:
            rel_imp = (height / initial_detected * 100) if initial_detected != 0 else 0
            label = f'{height:.2f}\n({rel_imp:+.2f}%)'
        else:
            label = f'{height:.2f}\n(+0.0%)'
        if height >= 0:
            va = 'bottom'
            offset = (y_max - y_min) * 0.02
        else:
            va = 'top'
            offset = -(y_max - y_min) * 0.02
        axes[i, 1].text(
            bar.get_x() + bar.get_width()/2,
            height + offset,
            label,
            ha='center',
            va=va,
            fontsize=12
        )

    if i == n_projects - 1:
        axes[i, 0].set_xticks(x_positions)
        axes[i, 0].set_xticklabels([format_variant_label(v) for v in variants], rotation=45, ha='right')
        axes[i, 1].set_xticks(x_positions_imp)
        axes[i, 1].set_xticklabels([format_variant_label(v) for v in improvement_variants], rotation=45, ha='right')
    else:
        axes[i, 0].set_xticks([])
        axes[i, 1].set_xticks([])

    axes[i, 0].set_yticks([])
    axes[i, 1].set_yticks([])

legend_elements = [
    Patch(facecolor=colors[0], label='INITIAL'),
    Patch(facecolor=colors[1], label='NAIVE Variants'),
    Patch(facecolor=colors[2], label='IMPROVED Variants')
]

fig.legend(handles=legend_elements, loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.03))

plt.tight_layout()
plt.savefig('fig_mutation_detection_comparison.pdf', format='pdf', dpi=300, bbox_inches='tight')
plt.show()

## Percentage of Survived / Detected / ... Mutants per Project + Variant + Mutator

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT
    mr.project_id, mr.project_name, mr.variant, mr.mutator,
    mr.survived_of_covered_pct, mr.killed_of_covered_pct, mr.timed_out_of_covered_pct, mr.memory_error_of_covered_pct, mr.run_error_of_covered_pct,
    mr.survived_of_covered_pct_diff, mr.killed_of_covered_pct_diff, mr.timed_out_of_covered_pct_diff, mr.memory_error_of_covered_pct_diff, mr.run_error_of_covered_pct_diff
FROM mv_mutation_results_by_project_variant_mutator mr
JOIN v_projects_successes ps ON ps.project_id = mr.project_id
WHERE mr.variant IN ('IMPROVED_200_TRIES')
""", conn)

df = df.rename(columns={
    'survived_of_covered_pct': 'Survived (%)',
    'killed_of_covered_pct': 'Killed (%)',
    'timed_out_of_covered_pct': 'Timed-Out (%)',
    'memory_error_of_covered_pct': 'Memory Error (%)',
    'run_error_of_covered_pct': 'Run Error (%)',
    'survived_of_covered_pct_diff': 'Survived (%) Diff.',
    'killed_of_covered_pct_diff': 'Killed (%) Diff.',
    'timed_out_of_covered_pct_diff': 'Timed-Out (%) Diff.',
    'memory_error_of_covered_pct_diff': 'Memory Error (%) Diff.',
    'run_error_of_covered_pct_diff': 'Run Error (%) Diff.',
})

df

## Overview of Mutation Scores and Changes per Project + Variant

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

def fetch_mutation_data(conn, variants=None):
    """Fetch and prepare mutation data from database."""
    query = "SELECT mr.* FROM mv_mutation_results_by_project_variant_mutator mr JOIN v_projects_successes ps ON ps.project_id = mr.project_id"
    if variants:
        variant_list = "', '".join(variants)
        query += f" WHERE variant IN ('{variant_list}')"

    results = pd.read_sql_query(query, conn)

    # Rename columns to match visualization expectations
    return results.rename(columns={
        'detected_pct': 'detection_rate',
        'detected_pct_diff': 'improvement'
    })

def get_sorted_variants(mutator_results, conn):
    """Get variants sorted by their order from database."""
    all_variants = sorted(
        mutator_results['variant'].unique(), 
        key=lambda v: pd.read_sql_query(f"SELECT variant_order('{v}')", conn).iloc[0, 0]
    )
    improvement_variants = [v for v in all_variants if v not in ['ORIGINAL', 'INITIAL', 'BASELINE']]
    return all_variants, improvement_variants

def create_variant_color_mapping(variants):
    """Create a consistent color mapping for variants using seaborn's colorblind palette."""
    # Use seaborn's colorblind-friendly palette
    color_palette = sns.color_palette("colorblind", n_colors=len(variants))

    # Convert RGB tuples to hex codes using matplotlib's color conversion
    hex_palette = [mcolors.to_hex(color) for color in color_palette]

    return {variant: hex_palette[i] for i, variant in enumerate(variants)}

def calculate_y_axis_limits(mutator_results, project_ids):
    """Calculate consistent y-axis limits across all plots."""
    detection_y_max = 0
    improvement_y_max = 0
    improvement_y_min = 0

    for project_id in project_ids:
        project_data = mutator_results[mutator_results['project_id'] == project_id]
        detection_y_max = max(detection_y_max, project_data['detection_rate'].max() * 1.1)

        project_improvements = project_data[project_data['improvement'].notna()]['improvement']
        if not project_improvements.empty:
            improvement_y_max = max(improvement_y_max, project_improvements.max() * 1.1)
            improvement_y_min = min(improvement_y_min, project_improvements.min() * 1.1)

    return detection_y_max, improvement_y_max, improvement_y_min

def plot_detection_rates(ax, project_data, all_variants, all_mutators, variant_colors):
    """Plot detection rate bars for a project."""
    mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}
    width = 0.8 / len(all_variants)
    total_width = width * len(all_variants)
    group_offsets = -total_width/2 + width/2

    for variant_idx, variant in enumerate(all_variants):
        variant_data = project_data[project_data['variant'] == variant]
        detection_rates = {row['mutator']: row['detection_rate'] for _, row in variant_data.iterrows()}

        for mutator, pos in mutator_positions.items():
            rate = detection_rates.get(mutator, 0)
            if rate > 0:
                offset = group_offsets + width * variant_idx
                ax.bar(pos + offset, rate, width, color=variant_colors[variant])

def plot_improvements(ax, project_data, improvement_variants, all_mutators, variant_colors):
    """Plot improvement bars for a project."""
    if not improvement_variants:
        return

    mutator_positions = {mutator: i for i, mutator in enumerate(all_mutators)}
    width = 0.8 / len(improvement_variants)
    total_width = width * len(improvement_variants)
    group_offsets = -total_width/2 + width/2

    for variant_idx, variant in enumerate(improvement_variants):
        variant_data = project_data[project_data['variant'] == variant]
        improvements = {row['mutator']: row['improvement'] 
                       for _, row in variant_data.iterrows() 
                       if row['improvement'] is not None}

        for mutator, pos in mutator_positions.items():
            impr = improvements.get(mutator, 0)
            if impr != 0:
                offset = group_offsets + width * variant_idx
                ax.bar(pos + offset, impr, width, color=variant_colors[variant])

def configure_axis(ax, title, ylabel, x_min, x_max, y_min, y_max, all_mutators, show_xticklabels=False):
    """Configure axis properties."""
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xticks(np.arange(len(all_mutators)))

    if show_xticklabels:
        ax.set_xticklabels(all_mutators, rotation=90, ha='center')
    else:
        ax.set_xticklabels([])

    ax.grid(axis='y', linestyle='--', alpha=0.7)

# Main visualization function
def visualize_mutation_results(conn, variants=None):
    """Create comprehensive visualization of mutation testing results."""
    # Set seaborn style for better aesthetics
    sns.set_style("whitegrid")

    # Fetch and prepare data
    mutator_results = fetch_mutation_data(conn, variants)
    all_variants, improvement_variants = get_sorted_variants(mutator_results, conn)
    all_mutators = sorted(mutator_results['mutator'].unique())
    project_ids = mutator_results['project_id'].unique()
    project_count = len(project_ids)

    # Calculate axis limits
    detection_y_max, improvement_y_max, improvement_y_min = calculate_y_axis_limits(
        mutator_results, project_ids)

    # Create color mapping using seaborn's colorblind palette
    variant_colors = create_variant_color_mapping(all_variants)

    # Create figure and grid
    fig = plt.figure(figsize=(18, 5 + 4 * project_count))
    gs = GridSpec(project_count, 2, width_ratios=[3, 2])

    # Set fixed x-axis limits
    x_min, x_max = -0.5, len(all_mutators) - 0.5

    # Create legend
    legend_handles = [plt.Rectangle((0, 0), 1, 1, color=variant_colors[variant]) 
                     for variant in all_variants]
    fig.legend(
        legend_handles, all_variants, loc='upper center',
        ncol=min(len(all_variants), 5), bbox_to_anchor=(0.5, 0.98), fontsize='small'
    )

    # Create plots for each project
    for i, project_id in enumerate(project_ids):
        project_data = mutator_results[mutator_results['project_id'] == project_id]
        if project_data.empty:
            continue

        # Get project name
        project_name = pd.read_sql_query(
            f"SELECT project_name({project_id})", conn).iloc[0, 0]

        # Create subplots
        ax1 = fig.add_subplot(gs[i, 0])  # Detection rate plot
        ax2 = fig.add_subplot(gs[i, 1])  # Improvement plot

        # Plot detection rates
        plot_detection_rates(ax1, project_data, all_variants, all_mutators, variant_colors)
        configure_axis(
            ax1, f'Detection Rate - Project ID: {project_id} - {project_name}',
            'Detection Rate (%)', x_min, x_max, 0, detection_y_max,
            all_mutators, show_xticklabels=(i == project_count - 1)
        )

        # Plot improvements
        plot_improvements(ax2, project_data, improvement_variants, all_mutators, variant_colors)
        configure_axis(
            ax2, f'Improvement - Project ID: {project_id} - {project_name}',
            'Improvement (%)', x_min, x_max, improvement_y_min, improvement_y_max,
            all_mutators, show_xticklabels=(i == project_count - 1)
        )
        ax2.axhline(y=0, color='k', linestyle='-', alpha=0.3)

    # Adjust layout
    plt.tight_layout(rect=[0, 0.03, 1, 0.92])
    plt.subplots_adjust(hspace=0.3, top=0.88)

    return fig

variants_to_plot = ['INITIAL', 'NAIVE_200_TRIES', 'IMPROVED_200_TRIES']
fig = visualize_mutation_results(conn, variants_to_plot)
plt.show()


## Number of newly killed mutants per project + variant

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT *
FROM mv_generalization_effects
""", conn)

df[['project_id', 'project_name', 'a_variant', 'b_variant', 'killed_mutations']]

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Filter data for the two plots
original_df = df[df['a_variant'] == 'ORIGINAL']
initial_df = df[df['a_variant'] == 'INITIAL']

# Function to create the grouped bar chart with consistent colors
def create_grouped_bar_chart(data, title):
    if data.empty:
        print(f"No data available for: {title}")
        return None

    # Sort by project_id and b_variant_order
    data = data.sort_values(['project_id', 'b_variant_order'])

    plt.figure(figsize=(16, 8))

    # Create combined project identifier (id + name)
    data['project_label'] = data['project_id'].astype(str) + ': ' + data['project_name']

    # Get unique project labels and variants
    project_labels = data['project_label'].unique()
    variants = data.sort_values('b_variant_order')['b_variant'].unique()

    # Create a color map for variants (using the new approach)
    color_map = plt.colormaps['tab10']
    variant_colors = {variant: color_map(i % 10) for i, variant in enumerate(variants)}

    # Set up the plot
    ax = plt.subplot(111)
    bar_width = 0.8 / len(variants)

    # For each project, plot bars for each variant
    for i, project_label in enumerate(project_labels):
        project_data = data[data['project_label'] == project_label]

        # For each variant, find its data for this project
        for j, variant in enumerate(variants):
            variant_data = project_data[project_data['b_variant'] == variant]

            # Calculate bar position
            x_pos = i + (j * bar_width) - (len(variants) * bar_width / 2) + (bar_width / 2)

            # If we have data for this variant in this project
            if not variant_data.empty:
                value = variant_data['killed_mutations'].values[0]

                # Plot the bar with consistent color
                bar = ax.bar(x_pos, value, width=bar_width,
                       color=variant_colors[variant],
                       label=variant if i == 0 else "")

                # Add text on top of the bar
                ax.text(x_pos, value + 0.1, str(int(value)),
                        ha='center', va='bottom', fontsize=9)
            else:
                # Plot an empty/zero bar to maintain spacing
                ax.bar(x_pos, 0, width=bar_width, color=variant_colors[variant],
                      label=variant if i == 0 else "")

    # Set x-axis labels and ticks
    ax.set_xlabel('Project')
    ax.set_ylabel('Newly Killed Mutations')
    ax.set_title(title)
    ax.set_xticks(range(len(project_labels)))
    ax.set_xticklabels(project_labels, rotation=45, ha='right')

    # Create legend with unique entries
    handles, labels = plt.gca().get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    plt.legend(by_label.values(), by_label.keys(), title='Variant')

    # Add a bit of padding to the top to accommodate the numbers
    y_max = data['killed_mutations'].max() if not data.empty else 10
    plt.ylim(0, y_max * 1.1)

    plt.tight_layout()
    return plt

# Create the two plots
plot1 = create_grouped_bar_chart(original_df, 'Newly Killed Mutations by Project (a_variant = ORIGINAL)')
if plot1 is not None:
    plot1.show()

plot2 = create_grouped_bar_chart(initial_df, 'Newly Killed Mutations by Project (a_variant = INITIAL)')
if plot2 is not None:
    plot2.show()


## Effects of Generalization on Test Suite Size + Runtime

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT *
FROM mv_generalization_effects
WHERE a_variant = 'ORIGINAL' AND b_variant IN ('IMPROVED_200_TRIES', 'NAIVE_200_TRIES')
""", conn)

In [ ]:
df[['project_name', 'a_variant', 'b_variant', 'tests_before', 'added_tests', 'removed_tests', 'tests_after', 'tests_delta', 'tests_delta_pct']]

In [ ]:
from natsort import natsorted

# Custom sort orders
variant_order = {
    'NAIVE_200_TRIES': 0,
    'IMPROVED_200_TRIES': 1
}
project_order = {
    'eqbench-es-default-1s': 0,
    'eqbench-es-default-10s': 1,
    'eqbench-es-default-60s': 2,
    'commons-utils': 3,
    'commons-utils-es-default-1s': 4,
    'commons-utils-es-default-10s': 5,
    'commons-utils-es-default-60s': 6
}

# Prepare and sort the DataFrame
df_tests = df[['b_variant', 'project_name', 'tests_before', 'added_tests', 'removed_tests', 'tests_after', 'tests_delta', 'tests_delta_pct']]

df_tests = df_tests.reindex(
    index=natsorted(
        df_tests.index,
        key=lambda x: (
            variant_order.get(df_tests.loc[x, 'b_variant'], 99),
            project_order.get(df_tests.loc[x, 'project_name'], 99),
            df_tests.loc[x, 'project_name']
        )
    )
)

# Variant LaTeX macro mapping
variant_map = {
    'IMPROVED_200_TRIES': r'\VariantImprovedC{}',
    'NAIVE_200_TRIES': r'\VariantNaiveC{}'
}

def project_group(project_name):
    if project_name.startswith('eqbench'):
        return 'eqbench'
    elif project_name.startswith('commons-utils'):
        return 'commons-utils'
    else:
        return 'other'

# Format and build LaTeX table rows with midrules between project groups
latex_rows = []
prev_group = None
for _, row in df_tests.iterrows():
    current_group = project_group(row['project_name'])
    if prev_group is not None and current_group != prev_group:
        latex_rows.append(r'\midrule')
    prev_group = current_group

    variant = variant_map.get(row['b_variant'], row['b_variant'])
    project = row['project_name']
    before = int(row['tests_before'])
    added = int(row['added_tests'])
    removed = int(row['removed_tests'])
    after = int(row['tests_after'])
    delta = '--' if row['tests_delta'] == 0 else f"{'+' if row['tests_delta'] >= 0 else ''}{int(row['tests_delta'])}"
    delta_pct = (
        '--'
        if row['tests_delta_pct'] == 0
        else f"{'+' if row['tests_delta_pct'] >= 0 else ''}{row['tests_delta_pct']:.2f} \\%"
    )
    latex_row = f"{variant} & {project} & {before} & {added} & {removed} & {after} & {delta} & {delta_pct} \\\\"
    latex_rows.append(latex_row)

# Build the full LaTeX table
latex_table = r"""\begin{table}[H]
\caption{Number of tests before and after generalization, with changes, per project.}
\label{tab:tests-per-project}
\begin{tabular}{llrrrrrr}
\toprule
 & & \multicolumn{6}{c}{Tests} \\
\cmidrule(lr){3-8}
Variant & Project & Before & Added & Removed & After & Delta & Delta \% \\
\midrule
""" + "\n".join(latex_rows) + r"""
\bottomrule
\end{tabular}
\end{table}
"""

print(latex_table)


In [ ]:
df[['project_name', 'a_variant', 'b_variant', 'lines_before', 'added_lines', 'removed_lines', 'lines_after', 'lines_delta', 'lines_delta_pct']]

In [ ]:
from natsort import natsorted

# Table 2: Lines
df_lines = df[['project_name', 'lines_before', 'added_lines', 'removed_lines', 'lines_after', 'lines_delta', 'lines_delta_pct']]
df_lines = df_lines.reindex(index=natsorted(df_lines.index, key=lambda x: df_lines.loc[x, 'project_name']))
df_lines = df_lines.rename(columns={
    'project_name': 'Project',
    'lines_before': 'Lines Before',
    'added_lines': 'Added',
    'removed_lines': 'Removed',
    'lines_after': 'After',
    'lines_delta': 'Delta',
    'lines_delta_pct': 'Delta %'
})

df_lines['Delta'] = df_lines['Delta'].apply(lambda x: f"{'+' if x >= 0 else ''}{x}")
df_lines['Delta %'] = df_lines['Delta %'].apply(lambda x: f"{'+' if x >= 0 else ''}{x:.1f} %")

display(df_lines)

# Generate LaTeX table body (without header)
latex_body_lines = df_lines.to_latex(
    index=False,
    header=False,
    escape=True,
    column_format='lrrrrrr',
    float_format='{:.2f}'.format
)

# Build the full LaTeX table with a multicolumn header
latex_table_lines = r"""\begin{table}[H]
\caption{Number of test lines before and after generalization, with changes, per project.}
\label{tab:lines-per-project}
\begin{tabular}{lrrrrrr}
\toprule
 & \multicolumn{6}{c}{Lines} \\
\cmidrule(lr){2-7}
Project & Before & Added & Removed & After & Delta & Delta \% \\
\midrule
""" + "\n".join(latex_body_lines.splitlines()[3:-2]) + r"""
\bottomrule
\end{tabular}
\end{table}
"""

print(latex_table_lines)

In [ ]:
df[['project_name', 'a_variant', 'b_variant', 'runtime_before', 'added_runtime', 'removed_runtime', 'runtime_after', 'runtime_delta', 'runtime_delta_pct']]

In [ ]:
from natsort import natsorted

# Table 3: Runtime
df_runtime = df[['project_name', 'runtime_before', 'added_runtime', 'removed_runtime', 'runtime_after', 'runtime_delta', 'runtime_delta_pct']]
df_runtime = df_runtime.reindex(index=natsorted(df_runtime.index, key=lambda x: df_runtime.loc[x, 'project_name']))
df_runtime = df_runtime.rename(columns={
    'project_name': 'Project',
    'runtime_before': 'Runtime Before',
    'added_runtime': 'Added',
    'removed_runtime': 'Removed',
    'runtime_after': 'After',
    'runtime_delta': 'Delta',
    'runtime_delta_pct': 'Delta %'
})

df_runtime['Delta'] = df_runtime['Delta'].apply(lambda x: f"{'+' if x >= 0 else ''}{x:.2f}")
df_runtime['Delta %'] = df_runtime['Delta %'].apply(lambda x: f"{'+' if x >= 0 else ''}{x:.1f} %")

display(df_runtime)

# Generate LaTeX table body (without header)
latex_body_runtime = df_runtime.to_latex(
    index=False,
    header=False,
    escape=True,
    column_format='lrrrrrr',
    float_format='{:.2f}'.format
)

# Build the full LaTeX table with a multicolumn header
latex_table_runtime = r"""\begin{table}[H]
\caption{Test suite runtime before and after generalization, with changes, per project.}
\label{tab:runtime-per-project}
\begin{tabular}{lrrrrrr}
\toprule
 & \multicolumn{6}{c}{Runtime (in seconds)} \\
\cmidrule(lr){2-7}
Project & Before & Added & Removed & After & Delta & Delta \% \\
\midrule
""" + "\n".join(latex_body_runtime.splitlines()[3:-2]) + r"""
\bottomrule
\end{tabular}
\end{table}
"""

print(latex_table_runtime)

## Description of Generalizations That Killed New Mutants

In [ ]:
import pandas as pd

df = pd.read_sql_query("""
SELECT *
FROM mv_mutation_status_changes
WHERE a_is_detected IS FALSE AND b_status = 'KILLED'
""", conn)

df

## Comparison of Detected vs. Undetected Mutants

In [ ]:
import re
import pandas as pd
from natsort import natsorted

df = pd.read_sql_query("""
SELECT * FROM mv_mutation_detection_comparison
""", conn)

# Replace True/False with 'yes'/'no'
df['is_detected'] = df['is_detected'].map({True: 'yes', False: 'no'})

# Set is_detected as categorical so 'yes' sorts before 'no'
df['is_detected'] = pd.Categorical(
    df['is_detected'],
    categories=['yes', 'no'],
    ordered=True
)

# Sort by project_name (natural sort) and is_detected ('yes' first)
projects_sorted = natsorted(df['project_name'].unique())
df['project_name'] = pd.Categorical(df['project_name'], categories=projects_sorted, ordered=True)
df = df.sort_values(by=['project_name', 'is_detected'])

# --- Calculate percent of mutants per project ---
project_totals = df.groupby('project_name', observed=False)['count'].transform('sum')
df['mutant_percent'] = (df['count'] / project_totals * 100)

display(df)

# Create the LaTeX table
def safe_format(val, fmt="{:.0f}"):
    if pd.isna(val):
        return '-'
    return fmt.format(val)

def phantom_pad(num, width=3):
    if pd.isna(num):
        return '-'
    s = f"{int(num)}"
    return r"\phantom{0}" * (width - len(s)) + s

def get_base_project(name):
    return re.sub(r'-\d+s$', '', name)

lines = []
lines.append(r"\begin{table}[H]")
lines.append(r"  \caption{Model properties of mutants that are (not) detected by the IMPROVED$_{200}$ variant.}")
# lines.append(r"  %\caption{Comparison of un-/detected mutants of variant IMPROVED$_{200}$. Model Size shows the number of characters in the Java representation of the model.}")
lines.append(r"  \label{tab:mutation-detection-comparison}")
lines.append(r"  \begin{tabular}{lcrrrrr}")
lines.append(r"    \toprule")
lines.append(r"            &          &         & Model Size & Operations & Constraints & Constr. Used \% \\")
lines.append(r"    Project & Detected & Mutants & (Median) & (Median) & (Median) & (Mean / Median) \\")
lines.append(r"    \midrule")

if not df.empty:
    prev_base = df.iloc[0]['project_name']#get_base_project(df.iloc[0]['project_name'])
    for idx, row in df.iterrows():
        current_base = row['project_name']#get_base_project(row['project_name'])
        if idx != df.index[0] and current_base != prev_base:
            lines.append(r"    \midrule")
        prev_base = current_base

        # Mutants column with percentage
        mutants_str = row['count']
        # mutants_str = f"{row['count']} ({row['mutant_percent']:.1f} \\%)"

        lines.append(
            "    " +
            f"{row['project_name']} & " +
            f"{row['is_detected']} & " +
            f"{mutants_str} & " +
            f"{safe_format(row['median_model_java_size'])} & " +
            f"{safe_format(row['median_model_operation_count'])} & " +
            f"{safe_format(row['median_total_constraint_count'])} & " +
            f"{safe_format(row['avg_used_constraint_pct'])} \\% / {phantom_pad(row['median_used_constraint_pct'], width=3)} \\%" +
            r" \\"
        )

lines.append(r"    \bottomrule")
lines.append(r"  \end{tabular}")
lines.append(r"\end{table}")

latex_table = "\n".join(lines)
print(latex_table)
